In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.6 Relativistic Collisions and Decays

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume IV — Special Relativity",
    number="4.6",
    title="Relativistic Collisions and Decays",
    blurb="Four-momentum conservation, put to work: it fixes the energies of "
    "decay products, defines the invariant mass of a system, sets the "
    "energy needed to create new particles, and predicts how light "
    "scatters off matter.",
    difficulty="intermediate",
    estimate="120–150 min",
)

## Notebook overview

[§4.5](four-momentum-energy.ipynb) built the four-momentum and showed that its conservation is frame-independent.
This notebook turns that single law into a working tool for real particle physics. The
pattern never changes: **conserve the total four-momentum** $\sum p^\mu$, then use the
invariant $E^2-(pc)^2=(mc^2)^2$ to read off the physics. With nothing more than four-vector
arithmetic in `numpy`, we will fix the energies of decay products, define the invariant mass
that experimentalists use to *discover* particles, find the energy needed to create new
matter, and predict how light scatters off an electron.

Five problems carry the method. A particle decays, and conservation alone determines how its
energy is shared. A set of particles has an **invariant mass** — the norm of their combined
four-momentum — which is the same in every frame and shows up as a peak when a new particle
is found. The **center-of-momentum frame**, where the total spatial momentum vanishes, is the
natural place to analyze a collision. Creating new particles has a **threshold**, and for a
beam on a fixed target that threshold is higher than naive mass-counting suggests, because
momentum must be carried away — the calculation that set the size of the Bevatron. And
**Compton scattering**, the wavelength shift of light bouncing off an electron, falls straight
out of photon-plus-electron four-momentum conservation, the experiment that proved light
carries momentum and a bridge toward the quantum mechanics of Volume VI.

We reuse the four-vector and `np.einsum` toolkit of [§4.3](spacetime-minkowski.ipynb) and [§4.5](four-momentum-energy.ipynb) by cross-reference rather
than rebuilding it. Computations are in **SI units**, but particle masses and energies are
quoted in the field-standard $\mathrm{MeV}/c^2$ and MeV with the conversion stated, since the
numbers ($m_pc^2=938.3\,$MeV, the antiproton threshold $5630\,$MeV, the Compton wavelength
$2.43\,$pm) are the recognizable landmarks. Collisions are discrete before/after events with
nothing to animate; the key figures are a momentum schematic and two faithful plots.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the photon energies solved from the conservation equations; the system
> invariant mass equal to the parent mass and frame-independent; the $\Lambda$ daughters
> recoiling with equal momenta and reconstructing the parent; the COM frame having zero total
> momentum; the antiproton threshold derived numerically as $6m_pc^2$; the Compton recoil
> electron reconstructed on-shell. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.
>
> **Scope.** Relativistic kinematics via four-momentum conservation; dynamics in fields is
> [§4.7](relativistic-lagrangian-fields.ipynb). See Nolting, *Theoretical Physics 4* {cite}`nolting4`; Taylor & Wheeler, *Spacetime
> Physics* {cite}`taylor_wheeler`; Griffiths, *Introduction to Elementary Particles*; and
> [§4.5](four-momentum-energy.ipynb) (four-momentum and the invariant) and [§3.10](../03-electrodynamics/radiation.ipynb) (radiation carries momentum).

## Theory in brief

### Conservation of four-momentum

In any interaction the total four-momentum is conserved — energy and momentum together, in
every frame,

```{math}
:label: eq-four-momentum-conservation
\sum_{\rm initial} p^\mu = \sum_{\rm final} p^\mu .
```

Applied with the invariant $E^2-(pc)^2=(mc^2)^2$ of [§4.5](four-momentum-energy.ipynb), this one law solves a remarkable
range of problems by four-vector arithmetic.

### Particle decay

A parent of mass $M$ at rest has $p^\mu=(Mc,\mathbf 0)$, so its products must carry total
momentum zero and total energy $Mc^2$. For two products this fixes their energies uniquely,

```{math}
:label: eq-decay
M\to2\gamma:\ E_\gamma=\tfrac12 Mc^2, \qquad M\to m_1+m_2:\ E_1=\frac{(M^2+m_1^2-m_2^2)c^2}{2M} .
```

Conservation *determines* the kinematics; there is no freedom left.

### The invariant mass of a system

A set of particles has a total four-momentum whose norm defines the system's **invariant
mass**,

```{math}
:label: eq-invariant-mass-system
M_{\rm sys}c^2=\sqrt{E_{\rm tot}^2-(p_{\rm tot}c)^2},
```

the same in every frame. For decay products it equals the parent's mass, and is precisely the
quantity an experimentalist reconstructs — the "invariant-mass peak" that announces a particle.

### The center-of-momentum frame

The **COM frame** is the one where the total spatial momentum is zero. There the total energy
is the invariant mass times $c^2$,

```{math}
:label: eq-com-frame
\sum\mathbf p=\mathbf 0 \ \Rightarrow\ E_{\rm tot}^{\rm COM}=M_{\rm sys}c^2 .
```

One boosts into it, solves the kinematics there, and boosts back.

### Collision thresholds

To create new particles, the system's invariant mass must reach the total rest mass of the
products, all at rest in the COM at threshold. For a beam on a *stationary* target this costs
more than the products' rest energy, because momentum must be conserved,

```{math}
:label: eq-threshold
p+p\to p+p+p+\bar p:\quad T_{\rm threshold}=6\,m_pc^2 \ (\text{not } 2\,m_pc^2) .
```

The extra factor is the kinetic energy the products must keep — the number the Bevatron was
built to reach.

### Compton scattering

A photon scattering off a free electron shifts wavelength by

```{math}
:label: eq-compton
\Delta\lambda=\frac{h}{m_ec}\,(1-\cos\theta),
```

where $h/m_ec=2.43\,$pm is the **Compton wavelength**. It follows from four-momentum
conservation with the massless-photon relation $E=pc$ of [§4.5](four-momentum-energy.ipynb), and proved that light carries
momentum like a particle — a forward pointer to quantum mechanics.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from ecp import draw, validate

MU0 = 4.0e-7 * np.pi  # vacuum permeability, T·m/A
EPS0 = 8.8541878128e-12  # vacuum permittivity, F/m
C_LIGHT = 1.0 / np.sqrt(MU0 * EPS0)  # speed of light, m/s
H_PLANCK = 6.62607015e-34  # Planck constant, J·s
EV = 1.602176634e-19  # one electron-volt, J
MEV = 1.0e6 * EV  # one MeV, J
M_E = 9.1093837015e-31  # electron mass, kg  (m_e c^2 = 0.511 MeV)
M_P = 1.67262192369e-27  # proton mass, kg   (m_p c^2 = 938.3 MeV)
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# The Minkowski metric and four-vector norm — the toolkit of §4.3/§4.5, reused here.
ETA = np.diag([-1.0, 1.0, 1.0, 1.0])  # η = diag(−1, 1, 1, 1), signature (−,+,+,+)


def four_norm(p):
    """The Minkowski norm η_μν p^μ p^ν of a four-vector (eq-invariant-mass-system).

    The §4.3/§4.5 toolkit, reused: the contraction ``np.einsum('a,ab,b->', p, ETA, p)`` with the
    metric η = diag(-1, 1, 1, 1). For a four-momentum it returns -(mc)^2.

    Parameters
    ----------
    p : numpy.ndarray
        A length-4 array (a four-vector), components (E/c, p_x, p_y, p_z).

    Returns
    -------
    float
        The scalar norm η_μν p^μ p^ν.
    """
    return float(np.einsum("a,ab,b->", p, ETA, p))


def invariant_mass(p_total):
    """The invariant mass M = √(E^2 - (pc)^2)/c^2 of a (total) four-momentum (eq-invariant-mass-system).

    Computed from the Minkowski norm as M = √(-η_μν p^μ p^ν)/c, the
    frame-independent "length" of the four-momentum.

    Parameters
    ----------
    p_total : numpy.ndarray
        A length-4 four-momentum (or the sum of several), in kg·m/s.

    Returns
    -------
    float
        The invariant mass, in kg.
    """
    return np.sqrt(-four_norm(p_total)) / C_LIGHT


def momentum_of(E, m):
    """The spatial momentum magnitude p = √(E^2 - (mc^2)^2)/c from energy and mass (eq-decay).

    The rearranged invariant E^2 - (pc)^2 = (mc^2)^2 of §4.5, giving |p| for a particle
    of total energy ``E`` and rest mass ``m``.

    Parameters
    ----------
    E : float
        Total energy, in J.
    m : float
        Rest mass, in kg.

    Returns
    -------
    float
        The spatial momentum magnitude, in kg·m/s.
    """
    return np.sqrt(E**2 - (m * C_LIGHT**2) ** 2) / C_LIGHT


def boost4(beta):
    """The 4×4 Lorentz boost along x at β = v/c (the §4.3/§4.5 helper, reused).

    Parameters
    ----------
    beta : float
        Boost speed as a fraction of c (|β| < 1).

    Returns
    -------
    numpy.ndarray
        A ``(4, 4)`` array acting on a four-vector (E/c, p_x, p_y, p_z).
    """
    g = 1.0 / np.sqrt(1.0 - beta**2)
    gb = g * beta
    return np.array(
        [
            [g, -gb, 0.0, 0.0],
            [-gb, g, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ]
    )

## Exercise 1 — Decay into two photons

Begin with the simplest decay. A neutral pion ($\pi^0$, mass $135\,\mathrm{MeV}/c^2$) at rest
decays into two photons, $\pi^0\to2\gamma$. The parent's four-momentum is $(Mc,\mathbf 0)$, so
four-momentum conservation {eq}`eq-four-momentum-conservation` forces the two photons to fly
off **back-to-back** with equal energies, and that shared energy must be exactly half the rest
energy, $E_\gamma=\tfrac12Mc^2$ {eq}`eq-decay` ({numref}`fig-cd-decay`). Conservation alone
fixes everything.

**This exercise (worked).** Treat the two photon energies as unknowns and let conservation
find them: energy demands $E_1+E_2=Mc^2$, and momentum (with $p=E/c$ along $\pm x$) demands
$E_1-E_2=0$. Solve the two-equation linear system with `numpy.linalg.solve`, confirm the
solution is $E_1=E_2=\tfrac12Mc^2$, then assemble the photon four-momenta $(E/c,\pm E/c,0,0)$
and confirm with `np.allclose` that their sum equals the parent's $(Mc,\mathbf 0)$ — energy
and momentum conserved together.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    p_final,
    p_parent,
    "four-momentum is conserved in the decay (energy and momentum together)",
    rtol=1e-9,
)
validate.close(
    np.array([E1_g, E2_g]),
    np.full(2, 0.5 * M_pi * C_LIGHT**2),
    "solving the conservation equations gives each photon E = Mc²/2",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The invariant mass of a system

Here is the concept that makes particle discovery possible. A *set* of particles has a total
four-momentum, and its norm defines an **invariant mass** $M_{\rm sys}c^2=\sqrt{E_{\rm tot}^2-
(p_{\rm tot}c)^2}$ {eq}`eq-invariant-mass-system`, the same in every frame. For the two photons
of Exercise 1 it must equal the parent pion's mass — and crucially, it stays equal even when we
view the photons from a moving frame, where their individual energies change. This is exactly
how an unstable particle is found: not seen directly, but reconstructed as a peak in the
invariant mass of its decay products.

**This exercise (worked).** Compute the two-photon system's invariant mass with the `np.einsum`
norm (the `invariant_mass` helper), confirm it equals the pion mass $M$, then boost both
photons into a frame moving at $0.6c$ (`boost4`, the matrix product `@`) and confirm the
invariant mass is unchanged though each photon's energy is not.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.array([M_sys, M_sys_boost]),
    np.array([M_pi, M_pi]),
    "the invariant mass of the decay products equals the parent mass, in any frame",
    rtol=1e-9,
)

## Exercise 3 — Decay into two unequal masses

When the two products have *different* masses the energy is shared unequally, but conservation
still fixes the split exactly. The classic example is the decay of a $\Lambda$ baryon ($1116\,
\mathrm{MeV}/c^2$) into a proton ($938\,\mathrm{MeV}/c^2$) and a pion ($140\,\mathrm{MeV}/c^2$).
Energy and momentum conservation give the daughter energy $E_1=(M^2+m_1^2-m_2^2)c^2/(2M)$
{eq}`eq-decay`, with $E_2$ following by symmetry, and the two must sum to the parent rest energy
$Mc^2$ while each stays above its own rest energy.

**This exercise (worked).** For $\Lambda\to p+\pi^-$, evaluate $E_1$ and $E_2$ from the formula
as explicit `numpy` arithmetic, then put the split to the test it must pass: the daughters
recoil back-to-back, so `momentum_of(E1, m1)` and `momentum_of(E2, m2)` must agree; assemble
the daughter four-momenta with those momenta and confirm with `invariant_mass` that their sum
reconstructs the parent mass $M$. Check also that $E_1+E_2=Mc^2$ and that each daughter energy
is at least its rest energy $m_ic^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    np.array([p1_mag / p2_mag, M_reconstructed / M_L]),
    np.ones(2),
    "the daughters recoil with equal momenta and reconstruct the parent mass",
    rtol=1e-9,
)
validate.close(
    E1 + E2,
    M_L * C_LIGHT**2,
    "the daughter energies sum to the parent rest energy",
    rtol=1e-9,
)
validate.check(
    E1 >= m1 * C_LIGHT**2 and E2 >= m2 * C_LIGHT**2,
    "each daughter energy is at least its rest energy",
)

## Exercise 4 — The center-of-momentum frame

Collisions are simplest to analyze in the **center-of-momentum frame**, where the total spatial
momentum vanishes {eq}`eq-com-frame`. Given a system with nonzero total momentum, the boost that
reaches the COM has $\beta_{\rm COM}=p_{\rm tot}c/E_{\rm tot}$, and in that frame the entire
energy budget is the invariant mass, $E_{\rm tot}^{\rm COM}=M_{\rm sys}c^2$ — no energy is "tied
up" in bulk motion. This is the natural workshop for collision kinematics: boost in, solve,
boost back.

**This exercise (worked).** Take two protons, one at rest and one with kinetic energy $2\,$GeV,
form the total four-momentum, boost by $\beta_{\rm COM}=p^1_{\rm tot}/p^0_{\rm tot}$ with
`boost4`, and confirm the total spatial momentum is zero there and the COM energy equals the
invariant mass times $c^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    p_tot_com[1],
    0.0,
    "the COM frame has zero total spatial momentum (residual ≪ the incoming momentum)",
    atol=1e-9 * p_tot[1],
)
validate.close(
    E_com,
    invariant_mass(p_tot) * C_LIGHT**2,
    "in the COM frame the total energy equals the invariant mass × c²",
    rtol=1e-9,
)

## Exercise 5 — Collision threshold: antiproton production

To create new matter in a collision, the system's invariant mass must reach the total rest mass
of the products {eq}`eq-threshold`. The 1955 discovery of the antiproton used the reaction
$p+p\to p+p+p+\bar p$ — four particles out, each of mass $m_p$, so the invariant mass must reach
$4m_pc^2$. The subtlety, and the reason the Bevatron had to be so large, is that for a beam on a
*stationary* target the products cannot all be at rest in the lab: total momentum must be
conserved, so they must keep moving. Working through the invariant mass, the required beam
kinetic energy is $T=6m_pc^2$, three times the naive $2m_pc^2$ one might guess from the new
particles' rest energy alone.

**This exercise (worked).** *Derive* the threshold rather than assert it: write the system's
`invariant_mass` as a function of the beam kinetic energy $T$ (beam plus stationary-target
four-momenta) and solve $M_{\rm sys}(T)=4m_p$ numerically with `scipy.optimize.brentq`.
Confirm the root is $T=6m_pc^2\approx5630\,$MeV, and that the invariant mass indeed reaches
$4m_p$ there.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    T_threshold,
    6.0 * M_P * C_LIGHT**2,
    "the numerically derived antiproton threshold is T = 6 m_p c² (not 2 m_p c²)",
    rtol=1e-6,
)
validate.close(
    M_reached,
    4.0 * M_P,
    "at the derived threshold the invariant mass reaches 4 m_p",
    rtol=1e-6,
)

## Exercise 6 — Compton scattering

A photon scattering off a free electron changes wavelength, and the shift follows entirely from
four-momentum conservation between the photon and the electron, using the massless relation
$E=pc$ of [§4.5](four-momentum-energy.ipynb). The result is $\Delta\lambda=(h/m_ec)(1-\cos\theta)$ {eq}`eq-compton`, where
$h/m_ec=2.43\,$pm is the **Compton wavelength** and $\theta$ is the scattering angle: no shift
straight ahead ($\theta=0$), one Compton wavelength at $90^\circ$, and the maximum of two at
backscatter ($\theta=180^\circ$) ({numref}`fig-cd-compton`). A classical wave would not change
wavelength on scattering; the shift is direct evidence that light carries momentum like a
particle.

**This exercise (worked).** For a $0.5\,$MeV incident photon, impose four-momentum
conservation to obtain the scattered energy $E'$ at each angle and convert to wavelengths with
$\lambda=hc/E$. Then verify conservation the hard way: reconstruct the recoil electron's
four-momentum $p_e'=p_\gamma+p_e-p_\gamma'$ (scattered photon at angle $\theta$ in the
$xy$-plane) and confirm with `invariant_mass` that the recoil is **on-shell** — invariant mass
exactly $m_e$ — at $\theta=0^\circ,90^\circ,180^\circ$. Confirm also that the backscatter
shift is twice the Compton wavelength.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    m_recoil,
    np.full(3, M_E),
    "the reconstructed recoil electron is on-shell (invariant mass m_e) at every angle",
    rtol=1e-9,
)
validate.close(
    delta_lambda[-1],
    2.0 * lambda_compton,
    "the backscatter (180°) shift is twice the Compton wavelength",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — A collider versus a fixed target

Threshold reasoning explains why modern machines collide *beams* instead of firing at a fixed
target. The quantity that matters is the invariant mass reachable, since that is what bounds the
masses one can create. For a beam of energy $E$ on a stationary target the invariant mass grows
only as $\sqrt{E}$ — most of the beam energy goes into the bulk motion of the products, not into
new mass. But for two beams of energy $E$ colliding head-on, the total momentum is already zero,
so the lab *is* the COM frame and the entire $2E$ becomes invariant mass, growing *linearly*
with $E$ {eq}`eq-com-frame` ({numref}`fig-cd-collider`). At high energy the collider wins
enormously.

**This exercise (student).** For a proton beam of energy $E$, compute the invariant mass against
(a) a stationary proton and (b) a second proton of energy $E$ head-on, each via the `np.einsum`
norm. Confirm the head-on invariant mass is $2E/c^2$ and that the fixed-target one scales as
$\sqrt{E}$ (so doubling $E$ multiplies it by $\sqrt2$, not $2$).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    M_collider,
    2 * E / C_LIGHT**2,
    "head-on collisions put all the energy into invariant mass (2E/c²)",
    rtol=1e-3,
)
validate.close(
    M_fixed_4E / M_fixed,
    2.0,
    "the fixed-target invariant mass scales as √E (quadrupling E doubles it)",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — One law, all the kinematics

Look back at the variety of problems just solved: a pion's decay, the invariant mass that finds
new particles, the energy sharing of unequal daughters, the center-of-momentum frame, the
antiproton threshold, the Compton shift, the case for colliders. Every one yielded to the same
two ingredients — conservation of the total four-momentum, and the invariant $E^2-(pc)^2=
(mc^2)^2$. Relativistic kinematics, for all its reputation, is four-vector bookkeeping: assemble
the four-momenta, conserve their sum, and contract with the metric to read off masses and
energies. The physics lives in those two lines.

**This exercise.** Close with the unifying check: confirm with `np.allclose` that across three
different problems — the two-photon decay, the two-proton system, and the antiproton-threshold
system — the total four-momentum's invariant mass is exactly what conservation demands
($M_\pi$, the COM energy $E_{\rm COM}/c^2$ measured by boosting in Exercise 4, and $4m_p$
respectively), the single quantity that carried every calculation.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    all_consistent,
    "one conservation law and one invariant solved decays, COM, thresholds, and scattering",
)

## Notebook summary

- **Conservation of four-momentum** {eq}`eq-four-momentum-conservation`, {eq}`eq-decay`: for
  $\pi^0\to2\gamma$ the photons are back-to-back with $E_\gamma=Mc^2/2=67.5\,$MeV, and
  $\sum p^\mu$ is conserved (energy and momentum together) by `np.allclose`.
- **Invariant mass** {eq}`eq-invariant-mass-system`: $\sqrt{E_{\rm tot}^2-(p_{\rm tot}c)^2}/c^2$
  of the decay products equals the parent mass and is unchanged by a $0.6c$ boost — the
  invariant-mass peak that finds particles. For $\Lambda\to p+\pi^-$ the daughter energies
  ($943$ and $173\,$MeV) sum to $Mc^2$.
- **The COM frame** {eq}`eq-com-frame`: boosting by $\beta_{\rm COM}=p_{\rm tot}c/E_{\rm tot}$
  zeroes the total momentum, and there $E_{\rm tot}=M_{\rm sys}c^2$.
- **Thresholds** {eq}`eq-threshold`: $p+p\to p+p+p+\bar p$ needs $T=6m_pc^2\approx5630\,$MeV —
  derived numerically by solving $M_{\rm sys}(T)=4m_p$ with `scipy.optimize.brentq` — three
  times the naive $2m_pc^2$, because momentum must be carried away: the Bevatron number.
- **Compton scattering** {eq}`eq-compton`: four-momentum conservation gives $\Delta\lambda=
  (h/m_ec)(1-\cos\theta)$, zero forward and $2\times2.43\,$pm at backscatter, the reconstructed
  recoil electron exactly on-shell — proof that light carries momentum. **Colliders** beat
  fixed targets because head-on invariant mass grows as $2E$, not $\sqrt{E}$.

One conservation law and one invariant solved every problem: assemble four-momenta, conserve the
sum, contract with the metric.

## Outlook

- **The relativistic Lagrangian and motion in fields ([§4.7](relativistic-lagrangian-fields.ipynb)).** Charges moving in an
  electromagnetic field, tying back to the field tensor $F^{\mu\nu}$ of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb).
- **Mandelstam variables** and the systematic kinematics of scattering (a pointer).
- **Particle physics in practice.** Thresholds, resonances, and the invariant-mass peak as a
  discovery tool — the method behind real detector analyses.
- **The bridge to quantum mechanics.** Compton scattering and photon momentum point straight to
  the quantum mechanics of Volume VI.
- **Cross-reference** [§4.5](four-momentum-energy.ipynb) (four-momentum and the invariant) and [§3.10](../03-electrodynamics/radiation.ipynb) (radiation carries momentum).

In [ ]:
from ecp.style import footer

footer()